# meshXYZ — Quad Training on ColabAutoregressive quad-mesh transformer. Trains a topology-aware decoder on a balancedtriangle/quad dataset from Objaverse (auto-quadrangulated with pymeshlab).**Runtime:** Runtime → Change runtime type → **T4 GPU****Design notes (robust to Colab kernel restarts):**- Each cell redefines its own paths/variables, so any cell can be re-run alone after a restart.- Run cells top to bottom on a fresh session. After a restart, just re-run from Cell 0.- Encoder weights are cached to Drive, so a restart restores them instantly instead of re-downloading 1.3 GB.**Colab Free gotchas:**- `--processes 1` for data prep (multiprocessing causes `double free` crashes on Free).- `batch_size = 4` (quad sequences are long; 32 OOMs the T4 in attention).- torch_cluster has no wheel for recent torch — the repo ships a pure-PyTorch FPS fallback.- GPU quota is per-day on Free; if `nvidia-smi` shows no GPU, switch account or wait.

## Cell 0 — Global config (re-run this first after any restart)

In [ ]:
import os# Everything downstream reads these. Re-run this cell after a kernel restart.GITHUB_REPO   = 'https://github.com/vixrino/meshXYZ.git'BRANCH        = 'main'REPO_DIR      = '/content/meshXYZ'ENCODER_FILE_ID = '1qX_YTMAE2tLFppJps3vKAWFbFZgE9CtQ'ENCODER_LOCAL   = '/content/encoder.pth'ENCODER_DRIVE   = '/content/drive/MyDrive/mesh_genai/encoder.pth'TRAIN_DIR  = '/content/data/train'VAL_DIR    = '/content/data/val'OUTPUT_DIR = '/content/runs/quad'DRIVE_DATA = '/content/drive/MyDrive/mesh_genai/data'CONFIG_PATH = f'{REPO_DIR}/config/config-quad.yaml'PATCHED_CFG = '/tmp/config-quad-patched.yaml'# Data prep knobsN_TRAIN    = 100      # Colab Free: keep small. Scale up on Pro / cluster.N_VAL      = 20MIX_RATIO  = 0.5      # fraction of meshes to quadrangulate; rest stay trianglePROCESSES  = 1        # 1 on Free (avoids double-free crash); 2-4 on Pro# Training knobsBATCH_SIZE    = 4         # quad sequences are long; 4 fits T4. Raise on bigger GPU.DRY_RUN_STEPS = 100       # set to None for a full training runRESUME_FROM_CKPT = None   # set to a Drive .ckpt path to resumeprint('Config loaded. REPO_DIR =', REPO_DIR)

## Cell 1 — GPU check + Drive mount

In [ ]:
import subprocess# GPU checkr = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],                   capture_output=True, text=True)if r.returncode != 0:    print('WARNING: No GPU found. Runtime -> Change runtime type -> T4 GPU.')    print('(Training on CPU is ~100x slower and not usable for real runs.)')else:    print('GPU:', r.stdout.strip())# Drive mountfrom google.colab import drivedrive.mount('/content/drive', force_remount=False)print('Drive mounted.')

## Cell 2 — Clone repo + install dependencies

In [ ]:
import os, subprocess# Clone (or pull if already present)if not os.path.isdir(REPO_DIR):    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', GITHUB_REPO, REPO_DIR])else:    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH])os.chdir(REPO_DIR)import sysif REPO_DIR not in sys.path:    sys.path.insert(0, REPO_DIR)# Dependencies!pip install -q lightning==2.2.* wandb dacite jaxtyping einops trimesh objaverse pymeshlab gdownprint('Repo ready at', os.getcwd())

## Cell 2b — pymeshlab quadrangulation smoke testVerifies the tri->quad filter works on this Colab's pymeshlab version before spending time on data prep.

In [ ]:
import pymeshlab, tempfile, osdef _make_icosphere_obj(path):    phi = (1 + 5**0.5) / 2    verts = [(-1,phi,0),(1,phi,0),(-1,-phi,0),(1,-phi,0),(0,-1,phi),(0,1,phi),             (0,-1,-phi),(0,1,-phi),(phi,0,-1),(phi,0,1),(-phi,0,-1),(-phi,0,1)]    faces = [(0,11,5),(0,5,1),(0,1,7),(0,7,10),(0,10,11),(1,5,9),(5,11,4),(11,10,2),             (10,7,6),(7,1,8),(3,9,4),(3,4,2),(3,2,6),(3,6,8),(3,8,9),(4,9,5),             (2,4,11),(6,2,10),(8,6,7),(9,8,1)]    with open(path,'w') as f:        for v in verts: f.write(f'v {v[0]} {v[1]} {v[2]}\n')        for fc in faces: f.write(f'f {fc[0]+1} {fc[1]+1} {fc[2]+1}\n')with tempfile.TemporaryDirectory() as tmp:    src = os.path.join(tmp,'ico.obj'); dst = os.path.join(tmp,'ico_quad.obj')    _make_icosphere_obj(src)    ms = pymeshlab.MeshSet(); ms.load_new_mesh(src)    try:        ms.meshing_tri_to_quad_by_smart_triangle_pairing()        ms.save_current_mesh(dst, save_textures=False)        with open(dst) as f: lines = f.readlines()        quads = [l for l in lines if l.startswith('f ') and len(l.split())==5]        print(f'pymeshlab OK — produced {len(quads)} quad faces on icosphere')        if quads: print('  sample quad:', quads[0].strip())    except Exception as e:        print('WARNING: quadrangulation filter failed:', e)

## Cell 3 — Encoder weights (cached to Drive)

In [ ]:
import os, shutil, subprocessif os.path.exists(ENCODER_LOCAL):    print('Encoder already in /content.')elif os.path.exists(ENCODER_DRIVE):    shutil.copy(ENCODER_DRIVE, ENCODER_LOCAL)    print('Encoder restored from Drive (no download needed).')else:    print('Downloading encoder (1.3 GB)...')    subprocess.run(['gdown', ENCODER_FILE_ID, '-O', ENCODER_LOCAL])    os.makedirs(os.path.dirname(ENCODER_DRIVE), exist_ok=True)    shutil.copy(ENCODER_LOCAL, ENCODER_DRIVE)    print('Encoder downloaded and backed up to Drive for next time.')import torchckpt = torch.load(ENCODER_LOCAL, map_location='cpu', weights_only=False)print('Encoder OK. Keys:', list(ckpt.keys())[:5])

## Cell 4 — Dataset prep (balanced tri/quad)Downloads Objaverse meshes and converts a fraction (`MIX_RATIO`) to quads, keeping the restas triangles. This produces a balanced dataset so the model learns the tri/quad distinction.Slow on Colab Free (~10-15 min for N=100, single process). Restores from Drive cache if present.

In [ ]:
import os, shutil, subprocess, globdef count_meshes(d):    return len(glob.glob(os.path.join(d, '**', '*.obj'), recursive=True))os.chdir(REPO_DIR)# Restore from Drive cache if a previous prep was backed uprestored = Falsedrive_train = os.path.join(DRIVE_DATA, 'train')if os.path.isdir(drive_train) and count_meshes(drive_train) >= N_TRAIN * 0.05:    if not os.path.isdir(TRAIN_DIR):        shutil.copytree(drive_train, TRAIN_DIR)    drive_val = os.path.join(DRIVE_DATA, 'val')    if os.path.isdir(drive_val) and not os.path.isdir(VAL_DIR):        shutil.copytree(drive_val, VAL_DIR)    restored = True    print(f'Restored from Drive cache: {count_meshes(TRAIN_DIR)} train, {count_meshes(VAL_DIR)} val')if not restored:    proc = subprocess.Popen(        ['python','-u','scripts/prep_objaverse.py',         '--n', str(N_TRAIN), '--out_dir', TRAIN_DIR, '--split','train',         '--processes', str(PROCESSES), '--mix_ratio', str(MIX_RATIO)],        cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)    for line in proc.stdout:        print(line, end='', flush=True)    proc.wait()    # Back up to Drive for next session    os.makedirs(DRIVE_DATA, exist_ok=True)    if os.path.isdir(drive_train): shutil.rmtree(drive_train)    shutil.copytree(TRAIN_DIR, drive_train)    print('Train data backed up to Drive.')print(f'\nTrain: {count_meshes(TRAIN_DIR)} meshes')

## Cell 4b — Validation splitCopies a few meshes from train to val (data leak, acceptable for a dry run). For a real run,generate a separate val split with prep_objaverse.py --split val.

In [ ]:
import os, shutil, globos.makedirs(VAL_DIR, exist_ok=True)if len(glob.glob(f'{VAL_DIR}/*.obj')) == 0:    # Copy a mix of meshes (sorted gives deterministic pick)    for f in sorted(glob.glob(f'{TRAIN_DIR}/*.obj'))[:5]:        shutil.copy(f, VAL_DIR)    print('Copied 5 meshes from train to val (dry-run only).')print(f'Val: {len(glob.glob(f"{VAL_DIR}/*.obj"))} meshes')

## Cell 4c — Dataset composition audit

In [ ]:
import globfiles = glob.glob(f'{TRAIN_DIR}/*.obj')total_tri = total_quad = tri_only = quad_only = mixed = 0for fp in files:    with open(fp) as f:        lines = [l for l in f if l.startswith('f ')]    tri  = sum(1 for l in lines if len(l.split()) == 4)    quad = sum(1 for l in lines if len(l.split()) == 5)    total_tri += tri; total_quad += quad    tot = tri + quad    if tot == 0: continue    qf = quad / tot    if qf < 0.05: tri_only += 1    elif qf > 0.95: quad_only += 1    else: mixed += 1tot_faces = total_tri + total_quadprint(f'Meshes: {len(files)}  (tri-only {tri_only}, mixed {mixed}, quad-dominant {quad_only})')if tot_faces:    qpct = total_quad / tot_faces * 100    print(f'Face-level: {total_tri} tri ({100-qpct:.0f}%) | {total_quad} quad ({qpct:.0f}%)')    print('Balanced' if 40 <= qpct <= 60 else 'Imbalanced (ok for dry run if both types present)')

## Cell 5 — Wandb login

In [ ]:
import wandbwandb.login()print('Wandb entity:', wandb.api.default_entity)

## Cell 6 — Training configPatches config-quad.yaml at runtime: encoder path, batch size, and dry-run step overrides.Set `DRY_RUN_STEPS = None` in Cell 0 for a full training run.

In [ ]:
import os, yamlos.chdir(REPO_DIR)with open(CONFIG_PATH) as f:    cfg = yaml.safe_load(f)cfg['encoder']['weights_path'] = ENCODER_LOCALcfg['training']['batch_size']  = BATCH_SIZEif DRY_RUN_STEPS is not None:    cfg['training']['max_steps']         = DRY_RUN_STEPS    cfg['training']['val_every_n_steps'] = max(10, DRY_RUN_STEPS // 5)    cfg['training']['viz_every_n_steps'] = max(10, DRY_RUN_STEPS // 2)    cfg['training']['save_every']        = max(10, DRY_RUN_STEPS // 2)with open(PATCHED_CFG, 'w') as f:    yaml.dump(cfg, f)mode = f'DRY RUN ({DRY_RUN_STEPS} steps)' if DRY_RUN_STEPS else 'FULL TRAINING'print(f'Mode       : {mode}')print(f'Batch size : {cfg["training"]["batch_size"]}')print(f'Max steps  : {cfg["training"]["max_steps"]}')print(f'Encoder    : {cfg["encoder"]["weights_path"]}')

## Cell 7 — Launch training (streamed logs)

In [ ]:
import os, subprocess, gc, torchos.chdir(REPO_DIR)os.makedirs(OUTPUT_DIR, exist_ok=True)gc.collect()if torch.cuda.is_available():    torch.cuda.empty_cache()else:    print('WARNING: no GPU available — training will be extremely slow on CPU.')cmd = ['python','-u','-m','src.train',       '--train_dir', TRAIN_DIR, '--val_dir', VAL_DIR,       '--config', PATCHED_CFG, '--output_dir', OUTPUT_DIR]if RESUME_FROM_CKPT:    cmd += ['--ckpt', RESUME_FROM_CKPT]print('Launching:', ' '.join(cmd))print('=' * 60)proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,                        stderr=subprocess.STDOUT, text=True, bufsize=1)for line in proc.stdout:    print(line, end='', flush=True)print('=' * 60)print('Exit code:', proc.wait())

## Cell 8 — Sanity check (read metrics from Wandb)Pulls the latest run's history. On a balanced dataset you should see:- `train/loss` decreasing- `train/loss_tri_pad` non-zero and moving (model sees triangles to predict TRI_PAD)- `train/face_type_acc` rising above 0 (model distinguishes tri vs quad)

In [ ]:
import wandbapi = wandb.Api()project = f'{wandb.api.default_entity}/mesh_genai'runs = api.runs(project, order='-created_at')run = runs[0]print(f'Run: {run.name} (state={run.state})')hist = run.history(keys=['train/loss','train/loss_tri_pad','train/loss_coord',                         'train/loss_eos','train/face_type_acc'], pandas=True)if hist.empty:    print('No history yet — run more steps.')else:    last = hist.iloc[-1]    for c in hist.columns:        if c.startswith('train/'):            print(f'  {c:<26s} {last[c]:.4f}')

## Cell 9 — Reset & cleanup (optional)Run between attempts: clears stale Python caches, frees GPU memory, kills zombie training processes.

In [ ]:
import os, shutil, gc, subprocess# Kill any zombie training processsubprocess.run(['pkill', '-9', '-f', 'src.train'])# Clear stale bytecode caches (prevents subprocess loading old code)for root, dirs, _ in os.walk(REPO_DIR):    for d in dirs:        if d == '__pycache__':            shutil.rmtree(os.path.join(root, d), ignore_errors=True)# Free GPUgc.collect()try:    import torch    if torch.cuda.is_available():        torch.cuda.empty_cache()        free, total = torch.cuda.mem_get_info()        print(f'GPU free: {free/1e9:.1f} / {total/1e9:.1f} GB')except Exception:    passprint('Cleanup done.')

## Cell 10 — GPU heartbeat (optional)Run periodically during long CPU-bound prep to keep the GPU session alive and avoid disconnects.

In [ ]:
import torchif torch.cuda.is_available():    x = torch.randn(2000, 2000, device='cuda')    print('GPU heartbeat:', float((x @ x.T).sum()))else:    print('No GPU to ping.')